# 06 — Micro-timing avancé : distributions & lien avec l'intensité (V2)

Deux approfondissements :
1. **Distributions complètes** — au lieu de résumer chaque subdivision par moyenne ± écart-type, on affiche **toutes les frappes** : on voit la vraie forme du nuage (serré ? asymétrique ? bimodal ?).
2. **Micro-timing ↔ intensité** — quand la musique chauffe (tempo monte, jeu plus intense), le placement change-t-il ? Le musicien reste-t-il aussi précis ? Frappe-t-il plus fort ?

## 0. Imports & préparation

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import utils_V2 as u
df = pd.read_csv('output/dunun_frappes_V2.csv'); d1 = df[df['code']=='D1'].copy()
d1['tempo'] = 60/(d1['dur']/4)                          # pulsation (bpm) = proxy d'intensité (intensité ~ tempo)
d1['dev']   = (d1['rel'] - (d1['SD']-1)/12) * d1['dur'] * 1000   # écart à la grille (ms)
PLAYED = lambda s: [sd for sd in range(1,13) if (s['SD']==sd).sum() > 20]

## 1. Distributions complètes — chaque frappe est un point
x = subdivision, y = écart de timing (ms), **couleur = tempo/intensité**, barre noire = moyenne.

À lire : nuage **serré verticalement** = très précis ; barre **loin de 0** = frappe en avance/retard (le swing) ; **couleurs mélangées de haut en bas** = le placement ne dépend pas de l'intensité.

In [ ]:
rng = np.random.RandomState(0)
fig, ax = plt.subplots(1, 3, figsize=(17, 5), sharey=True)
vmin, vmax = d1['tempo'].quantile(.05), d1['tempo'].quantile(.95)
for a, p in zip(ax, u.PIECES):
    s = d1[d1['piece']==p]
    for sd in PLAYED(s):
        g = s[s['SD']==sd]
        gg = g.sample(250, random_state=0) if len(g) > 250 else g
        x = sd + rng.uniform(-0.28, 0.28, len(gg))
        a.scatter(x, gg['dev'], c=gg['tempo'], cmap='inferno', vmin=vmin, vmax=vmax, s=9, alpha=.6, edgecolors='none')
        a.plot([sd-0.3, sd+0.3], [g['dev'].mean()]*2, color='k', lw=2)
    a.axhline(0, color='#888', lw=1, ls='--'); a.set(title=p, xlabel='subdivision', xticks=range(1,13), xlim=(0.3,12.7))
ax[0].set_ylabel('écart à la grille (ms)  += retard / −= avance')
sm = plt.cm.ScalarMappable(cmap='inferno', norm=plt.Normalize(vmin, vmax)); sm.set_array([])
cb = fig.colorbar(sm, ax=ax, pad=.01, fraction=.03); cb.set_label('tempo (bpm) = intensité')
plt.suptitle('Distributions complètes du micro-timing du dunun (D1) — chaque point = une frappe', y=1.02)
plt.savefig('output/06_dunun_distributions.png', dpi=130, bbox_inches='tight'); plt.show()

## 2. Le musicien reste-t-il précis quand ça accélère ?
On compare les cycles **lents** (début, « froid ») et **rapides** (fin, « chaud »).

In [ ]:
def prec(x):
    x = x.copy(); x['mr'] = x.groupby('SD')['rel'].transform('mean')
    return ((x['rel']-x['mr'])*x['dur']*1000).std()
rows = []
for p in u.PIECES:
    s = d1[d1['piece']==p]; med = s['dur'].median()
    froid, chaud = s[s['dur']>med], s[s['dur']<=med]
    rows.append({'rythme':p, 'précision_froid_ms':round(prec(froid)), 'précision_chaud_ms':round(prec(chaud)),
                 'volume_froid':round(froid['rms'].mean(),3), 'volume_chaud':round(chaud['rms'].mean(),3),
                 'Δvolume_%':round((chaud['rms'].mean()/froid['rms'].mean()-1)*100)})
display(pd.DataFrame(rows))
print('La précision ne se dégrade pas quand ça accélère (contrôle expert). Le volume, lui, dépend du rythme.')

## 3. Corrélations — timing et volume face au tempo

In [ ]:
rows = []
for p in u.PIECES:
    s = d1[d1['piece']==p]
    rows.append({'rythme':p,
                 'corr(tempo, |écart timing|)':round(s['tempo'].corr(s['dev'].abs()),2),
                 'corr(tempo, volume)':round(s['tempo'].corr(s['rms']),2)})
display(pd.DataFrame(rows))
print('|écart timing| ≈ indépendant du tempo → le swing est stable. Volume : monte avec le tempo pour Maraka, pas pour Woloso.')

## Bilan
- **Distributions serrées** : chaque subdivision forme un nuage étroit (~±20 ms) → le placement est délibéré, reproductible.
- **Le swing est indépendant de l'intensité** : les couleurs (tempo) se mélangent dans chaque nuage → le musicien garde le *même feel* qu'il joue lentement ou vite. C'est la démonstration visuelle du timing **catégoriel** de Polak.
- **La précision ne se dégrade pas** en accélérant (elle se resserre même pour Maraka) → contrôle expert.
- **L'intensité passe par le tempo** (et parfois le volume : Maraka +14 %), **pas par le timing**.

→ Le micro-timing du dunun est une **signature stable et précise**, distincte de la dynamique d'intensité du morceau.